# 06. cal01_base_v1.csv 만들기

원본 읽기 → LOT와 강종 연결 → 제외 표시 → 과거 파생컬럼 → 정비와 부품 연결 → 임시 0, 1 → 기준 CSV 저장 순서임.
05에서 만든 센서 후보와 사건 연결표를 읽고, 여기서 전체 345,390행의 기준표를 만듦. 기존 값과 라벨 조건은 유지함.


## 1. 경로와 사용할 도구

os는 경로, json은 계산 조건 기록, hashlib는 파일 내용의 해시 기록용임. pandas와 numpy로 표와 숫자를 계산함.


In [ ]:
import os
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

# 저장소 루트, notebooks, namelesspark 폴더에서 같은 data 폴더를 읽음.
if os.path.isfile(os.path.join('data', 'README.md')):
    ROOT = os.getcwd()
elif os.path.isfile(os.path.join('..', 'data', 'README.md')):
    ROOT = os.path.abspath('..')
else:
    ROOT = os.path.abspath(os.path.join('..', '..'))

DATA_DIR = os.path.join(ROOT, 'data')

ENC = 'utf-8-sig'

TEMP_CSV = os.path.join(DATA_DIR, 'T-CR1-CAL01_온도.csv')
EVENT_CSV = os.path.join(DATA_DIR, 'G-02_조업이벤트.csv')
REPAIR_CSV = os.path.join(DATA_DIR, 'G-01_정기수리캘린더.csv')
EQP_CSV = os.path.join(DATA_DIR, 'G-01_설비마스터.csv')
PART_CSV = os.path.join(DATA_DIR, 'G-01_부품마스터.csv')
MAINT_CSV = os.path.join(DATA_DIR, 'G-02_정비이력.csv')
PARTCHG_CSV = os.path.join(DATA_DIR, 'G-02_부품교체이력.csv')
WELD_CSV = os.path.join(DATA_DIR, 'T-CR1-WLD01_용접.csv')  # 팀 양식의 경로. 이번 계산에서는 읽지 않음.
PROCESSED = os.path.join(DATA_DIR, 'processed')
REVIEW = os.path.join(PROCESSED, 'review', 'v1')
SIGNALS = os.path.join(PROCESSED, 'analysis', 'signals', 'v1')
FEATURE_FILE = os.path.join(PROCESSED, 'features', 'v1', '06_features_1min.csv')
CANDIDATE_FILE = os.path.join(SIGNALS, '05_all_sensor_sustained_candidates.csv')
ROW_LINK_FILE = os.path.join(REVIEW, '05_domain_row_index.csv')


## 2. 원본과 05의 후보, 사건 연결표 읽기

원본 7개와 05의 검토 결과를 읽음. 파생컬럼은 이 노트북에서 직접 계산함.


In [ ]:
# 파일별 데이터프레임을 직접 읽음.
temp = pd.read_csv(TEMP_CSV, encoding=ENC)
event = pd.read_csv(EVENT_CSV, encoding=ENC)
repair = pd.read_csv(REPAIR_CSV, encoding=ENC)
eqp = pd.read_csv(EQP_CSV, encoding=ENC)
part = pd.read_csv(PART_CSV, encoding=ENC)
maint = pd.read_csv(MAINT_CSV, encoding=ENC)
partchg = pd.read_csv(PARTCHG_CSV, encoding=ENC)

# 날짜 컬럼만 변환함. 원본 행이나 결측은 바꾸지 않음.
temp['MEAS_DT'] = pd.to_datetime(temp['MEAS_DT'], errors='raise')
event['EVT_DT'] = pd.to_datetime(event['EVT_DT'], errors='raise')
repair['STRT_DT'] = pd.to_datetime(repair['STRT_DT'], errors='raise')
repair['END_DT'] = pd.to_datetime(repair['END_DT'], errors='raise')
eqp['INSTALL_DT'] = pd.to_datetime(eqp['INSTALL_DT'], errors='raise')
maint['DETC_DT'] = pd.to_datetime(maint['DETC_DT'], errors='raise')
maint['STRT_DT'] = pd.to_datetime(maint['STRT_DT'], errors='raise')
maint['END_DT'] = pd.to_datetime(maint['END_DT'], errors='raise')

# 원본 표는 이 묶음에도 보관함. 아래 정비와 조업 맥락 계산에 사용함.
sources = {'temp': temp, 'event': event, 'repair': repair, 'eqp': eqp,
           'part': part, 'maint': maint, 'partchg': partchg}

candidates = pd.read_csv(CANDIDATE_FILE, encoding=ENC, parse_dates=['START', 'END'])
row_links = pd.read_csv(ROW_LINK_FILE, encoding=ENC, parse_dates=['MEAS_DT'])


## 3. LOT의 실제 관측 구간과 강종 연결

LOT의 첫 관측시각과 CR1 강종변경 시각이 정확히 같은 기록만 연결함. 가까운 시각을 대신 붙이지 않음.
제외 전 시작시각을 남겨야 061 앞부분을 잘라도 강종과 경과시간 기준이 바뀌지 않음.


In [ ]:
lots = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), ROWS=("MEAS_DT", "size")
)

changes = event.loc[
    event["PLANT_CD"].eq("CR1") & event["EVT_TYPE"].eq("강종변경"),
    ["EVT_DT", "BEF_VAL", "AFT_VAL"],
].sort_values("EVT_DT")

lots = lots.merge(
    changes[["EVT_DT", "AFT_VAL"]], left_on="START", right_on="EVT_DT",
    how="left", validate="many_to_one",
).rename(columns={"AFT_VAL": "GRADE"}).drop(columns="EVT_DT")

lots["DURATION_H"] = (lots["END"] - lots["START"]).dt.total_seconds() / 3600

lot_context = lots[["LOT_NO", "GRADE", "START", "END", "ROWS", "DURATION_H"]].sort_values("START").reset_index(drop=True)


## 4. 사용자가 지정한 네 시간 구간 표시

원본 행은 유지하고 EXCLUDED만 표시함. 아래 네 조건 이외의 자동 제외 기준을 추가하지 않음.


In [ ]:
USER_EXCLUSIONS = [
    (240058, '2024-06-09 08:00:00', 'at_or_after', '사용자 지정: 노 내림 구간 제외'),
    (240061, '2024-06-15 02:30:00', 'before', '사용자 지정: 승온 구간 제외'),
    (240114, '2024-11-04 20:00:00', 'at_or_after', '사용자 지정: 정기수리 전 하강 구간 제외'),
    (240117, '2024-11-13 09:00:00', 'before', '사용자 지정: 정기수리 후 승온 구간 제외'),
]

marked = temp.copy()

marked['EXCLUDED'] = False
marked['EXCLUSION_REASON'] = ''
exclusion_rows = []

for lot, boundary, direction, reason in USER_EXCLUSIONS:
    boundary = pd.Timestamp(boundary)

    if direction == 'at_or_after':
        time_mask = marked['MEAS_DT'] >= boundary
    else:
        time_mask = marked['MEAS_DT'] < boundary

    hit = (marked['LOT_NO'] == lot) & time_mask

    marked.loc[hit, 'EXCLUDED'] = True
    marked.loc[hit, 'EXCLUSION_REASON'] = reason

    exclusion_rows.append({'LOT_NO': lot, 'boundary': boundary,
                           'direction': direction, 'excluded_rows': int(hit.sum())})

exclusions = pd.DataFrame(exclusion_rows)

grade_by_lot = lot_context.set_index('LOT_NO')['GRADE']

marked['GRADE'] = marked['LOT_NO'].map(grade_by_lot)

print(exclusions)


## 5. 파생컬럼에 사용할 관측만 선택

앞에서 표시한 제외 구간을 빼고, LOT별로 계산함. 원본 전체 행은 기준표에 그대로 남김.


In [ ]:
TARGETS = ['TC-Z3', 'OP-Z3', 'TC-Z4', 'OP-Z4', 'TC-Z5', 'OP-Z5', 'CP-Z4']
OP_CHANNELS = ['OP-Z3', 'OP-Z4', 'OP-Z5']

# 원본 temp는 그대로 두고 파생 계산에 쓸 행만 따로 가져옴.
feature_temp = marked.loc[
    ~marked['EXCLUDED'], ['MEAS_DT', 'LOT_NO'] + TARGETS
].sort_values(['LOT_NO', 'MEAS_DT']).copy()

lot_summary = lot_context.set_index('LOT_NO')[['START', 'END', 'ROWS', 'GRADE']]

lot_summary = lot_summary.rename(columns={
    'START': '관측시작', 'END': '관측끝', 'ROWS': '원본행수', 'GRADE': '강종',
})


## 6. LOT 안에서 1분 평균과 관측 수 계산

빈 값은 채우지 않음. 출력 0도 유효한 관측으로 셈. 과거 특징 계산을 위한 중간표임.


In [ ]:
minute_frames = []
count_frames = []
saturation_frames = []

for lot_no, lot_data in feature_temp.groupby('LOT_NO', sort=True):
    values = lot_data.set_index('MEAS_DT')[TARGETS]

    # LOT 밖의 자료와 섞이지 않도록 이 LOT에서만 1분씩 묶음.
    minute = values.resample('1min').mean()
    counts = values.resample('1min').count()

    # 빈 OP를 '포화 아님'으로 세지 않기 위해 빈 값은 그대로 남김.
    op_values = values[OP_CHANNELS]
    saturated = op_values.ge(100).where(op_values.notna())

    saturation_count = saturated.resample('1min').sum(min_count=1)

    # 제외 전 원본의 첫 관측이 포함된 분부터 지난 분 수임. 설비 가동 시간은 아님.
    first_minute = lot_summary.loc[lot_no, '관측시작'].floor('min')

    minute['LOT경과_분'] = (minute.index - first_minute).total_seconds() / 60
    minute['LOT_NO'] = lot_no

    minute['강종'] = lot_summary.loc[lot_no, '강종']

    counts['LOT_NO'] = lot_no
    saturation_count['LOT_NO'] = lot_no

    minute_frames.append(minute.reset_index())
    count_frames.append(counts.reset_index())
    saturation_frames.append(saturation_count.reset_index())

minute_data = pd.concat(minute_frames, ignore_index=True)
counts = pd.concat(count_frames, ignore_index=True)
saturation_counts = pd.concat(saturation_frames, ignore_index=True)

# 지정 구간 제외 후 남은 출력 0을 결측으로 바꾸지 않았는지 표시함.
display(feature_temp[OP_CHANNELS].eq(0).sum().rename('제외 후 남은 출력 0 관측 수'))
display(minute_data[['LOT_NO', 'MEAS_DT', '강종'] + TARGETS].head())


## 7. 현재 분 이전의 과거 특징 38개 계산

7센서에 1분 전, 5분 전, 직전 10분 평균과 표준편차, 직전 두 분 차이를 붙임. OP 3개의 직전 10분 포화율도 계산함. 현재 분은 포함하지 않음.


In [ ]:
feature_frames = []

for lot_no, lot_data in minute_data.groupby('LOT_NO', sort=True):
    data = lot_data.set_index('MEAS_DT').copy()

    lot_counts = counts.loc[counts['LOT_NO'].eq(lot_no)].set_index('MEAS_DT')

    lot_saturation = saturation_counts.loc[
        saturation_counts['LOT_NO'].eq(lot_no)
    ].set_index('MEAS_DT')

    for sensor in TARGETS:
        # 현재 분을 빼고 이전 분부터 계산함.
        past = data[sensor].shift(1)
        data[sensor + '_1분전'] = past
        data[sensor + '_5분전'] = data[sensor].shift(5)

        data[sensor + '_직전10분평균'] = past.rolling(10, min_periods=10).mean()
        data[sensor + '_직전10분표준편차'] = past.rolling(10, min_periods=10).std()

        data[sensor + '_직전두분차이'] = past.diff()

    for sensor in OP_CHANNELS:
        # 현재 분의 포화 여부도 입력에서 빼기 위함임.
        past_saturation = lot_saturation[sensor].shift(1)
        past_count = lot_counts[sensor].shift(1)

        saturated_sum = past_saturation.rolling(10, min_periods=10).sum()
        valid_sum = past_count.rolling(10, min_periods=10).sum()

        data[sensor + '_직전10분포화율'] = saturated_sum / valid_sum * 100

    feature_frames.append(data.reset_index())

features = pd.concat(feature_frames, ignore_index=True)

# 관측 수는 품질 확인용임. 예측할 현재 분의 관측 수를 입력으로 넣지 않음.
count_cols = {}

for sensor in TARGETS:
    count_cols[sensor] = sensor + '_관측수'

features = features.merge(
    counts.rename(columns=count_cols),
    on=['LOT_NO', 'MEAS_DT'],
    how='left',
)

display(features[['LOT_NO', 'MEAS_DT', 'TC-Z3', 'TC-Z3_1분전',
                  'TC-Z3_직전10분평균', 'OP-Z3_직전10분포화율']].head(12))


## 8. 중간 파생표 저장

`features/v1/06_features_1min.csv`는 1분 단위 중간표임. 최종 기준표는 뒤에서 원본 한 행당 한 행으로 만듦.


In [ ]:
# 파생 계산표는 별도 폴더에 두고, 기준표에는 과거 특징 38개만 연결함.
os.makedirs(os.path.dirname(FEATURE_FILE), exist_ok=True)

features.to_csv(FEATURE_FILE, index=False, encoding=ENC)

# 기존 v1과 같은 CSV 읽기 방식으로 값과 자료형을 맞춤.
features = pd.read_csv(FEATURE_FILE, encoding=ENC, parse_dates=['MEAS_DT'])


## 9. CAL01 정비와 부품 상세 연결

BM, TBM, CBM을 모두 가져옴. 부품교체 표는 WO_NO, 부품 이름은 PART_CD로 연결함.
정비 한 건에 부품 여러 개가 있으면 다음 셀에서 정비 한 행으로 묶음.


In [ ]:
CAL_TAG = "P1-CR1-CAL01"

events = maint.loc[maint["EQP_TAG"].eq(CAL_TAG)].copy()

parts = partchg.loc[partchg["WO_NO"].isin(events["WO_NO"])].copy()

parts = parts.merge(part, on="PART_CD", how="left", validate="many_to_one", indicator=True)

parts = parts.drop(columns="_merge")


### 9-1. 부품 여러 개를 정비 한 행으로 묶기

부품명, 사유는 문자열로 연결하고 품목 수와 수량은 따로 셈. 정비 표를 기준으로 left join하므로 교체 없는 정비도 남음.


In [ ]:
part_summaries = []

for wo_no, group in parts.groupby('WO_NO', sort=True):
    part_summaries.append({
        'WO_NO': wo_no, 'PART_RECORD_COUNT': len(group),
        'PART_CODES': ' | '.join(group['PART_CD'].astype(str)),
        'PART_NAMES': ' | '.join(group['PART_NM'].astype(str)),
        'PART_QTY_TOTAL': group['QTY'].sum(),
        'PART_CHANGE_REASONS': ' | '.join(group['CHG_RSN'].astype(str)),
    })

summary = pd.DataFrame(part_summaries, columns=[
    'WO_NO', 'PART_RECORD_COUNT', 'PART_CODES', 'PART_NAMES',
    'PART_QTY_TOTAL', 'PART_CHANGE_REASONS'])

events = events.merge(summary, on="WO_NO", how="left", validate="one_to_one")

events["HAS_PART_RECORD"] = events["PART_RECORD_COUNT"].notna()

for column in ["PART_RECORD_COUNT", "PART_QTY_TOTAL"]:
    events[column] = events[column].fillna(0).astype(int)

for column in ["PART_CODES", "PART_NAMES", "PART_CHANGE_REASONS"]:
    events[column] = events[column].fillna("")


### 9-2. 정비 작업시간과 정지 분량 구분

DOWN_MIN은 기록된 정지 분량임. 작업 시작, 끝 사이 어디서 실제로 멈췄는지는 알 수 없어 정지 구간을 역산하지 않음.


In [ ]:
# 작업 기간과 기록된 정지 분량은 다른 값임. 정지 시작, 종료를 역산하지 않음.
events["WORK_MIN"] = (events["END_DT"] - events["STRT_DT"]).dt.total_seconds() / 60

events["DOWN_RECORD_STATUS"] = "정지시간 미기록"
events.loc[events["DOWN_MIN"].eq(0), "DOWN_RECORD_STATUS"] = "해당 정비의 정지시간 0 기록"
events.loc[events["DOWN_MIN"].gt(0), "DOWN_RECORD_STATUS"] = "정지시간 기록 있음; 시점 미확정"

events["DOWN_TO_WORK_RATIO"] = events["DOWN_MIN"].div(events["WORK_MIN"].where(events["WORK_MIN"].gt(0)))

events["DOWN_EXCEEDS_WORK"] = events["DOWN_MIN"].astype("Float64").gt(events["WORK_MIN"].astype("Float64"))

events["STOP_INTERVAL_KNOWN"] = False  # 현재 원본에는 실제 정지 시작, 종료 컬럼이 없음.

details = maint.loc[maint["EQP_TAG"].eq(CAL_TAG)].merge(
    parts, on="WO_NO", how="inner", validate="one_to_many"
)

maintenance = events.sort_values(['DETC_DT', 'WO_NO']).reset_index(drop=True)

part_details = details.sort_values(['STRT_DT', 'WO_NO', 'PART_CD']).reset_index(drop=True)

print('정비:', len(maintenance), '/ 부품 상세:', len(part_details))


## 10. 임시 라벨과 검토 표시 준비

1은 강한 지속 신호와 적격 정비 기록의 시간 연관을 함께 만족한 행임. 0은 이번 규칙 미검출이며 정상 확정이 아님.
사용자 제외 행만 LABEL을 비움. 아래 문턱과 24시간은 실험 가정이며 공정 허용치가 아님.


In [ ]:
VERSION = "multisensor_domain_weak_v1_20261015"
AMPLITUDE_MULTIPLIER = 2.0
ASSOCIATION_HOURS = 24
CP_LOT_SIGMA_MULTIPLIER = 12.0
MIN_ELAPSED_SECONDS = 10
EQP_TAG = "P1-CR1-CAL01"
AMBIGUOUS_REMARK = r"원인\s*미상|확인\s*중|이상\s*없|재현\s*안|재점검\s*필요"
raw = marked

sensor_columns = []

for column in raw.columns:
    if column.startswith(('TC-', 'OP-')) and '_' not in column:
        sensor_columns.append(column)

sensor_columns += ["CLN-IN", "CP-Z4"]

n = len(raw)

times = pd.to_datetime(raw.MEAS_DT, errors="raise").reset_index(drop=True)

lot_values = raw.LOT_NO.to_numpy()

excluded = raw.EXCLUDED.to_numpy(dtype=bool)

output = raw[["LOT_NO", "MEAS_DT", "GRADE", "EXCLUDED"]].reset_index(drop=True).copy()

output["LABEL"] = pd.Series(np.zeros(n, dtype=np.int8), dtype="Int8")

output.loc[excluded, "LABEL"] = pd.NA
output["LABEL_STATUS"] = "임시 약지도; 현장 확정 정답 아님"
output.loc[excluded, "LABEL_STATUS"] = "사용자 지정 제외"
output["LABEL_REASON"] = "전체 21계측 검토에서 이번 신호+정비 규칙 미검출; 정상 확정 아님"
output.loc[excluded, "LABEL_REASON"] = "사용자 지정 시간 제외"

if "EXCLUSION_REASON" in raw:
    output.loc[excluded, "LABEL_REASON"] = raw.EXCLUSION_REASON.to_numpy()[excluded]

output["RULE"] = np.where(excluded, "user_excluded", "weak_rule_not_detected")

for column in ["EVIDENCE_WO", "EVIDENCE_MNT_TYPE", "EVIDENCE_SENSOR", "EVIDENCE_CANDIDATE_IDS", "ASSOCIATION_KIND"]:
    output[column] = ""

output["LABEL_VERSION"] = VERSION
output["MODEL_INPUT_ALLOWED"] = False
flags = {}

for flag_name in [
    'ANY_SENSOR_CANDIDATE', 'CONTROL_RESPONSE_CONTEXT', 'PERIODIC_CONTEXT',
    'LOT_BOUNDARY_CONTEXT', 'MAINT_WORK_OVERLAP', 'SCHEDULED_REPAIR_CONTEXT',
    'REPAIR_PREPOST_24H_CONTEXT', 'GRADE_CHANGE_NEARBY_30MIN',
    'CP_RETURN_TO_BASELINE_CONTEXT',
]:
    flags[flag_name] = np.zeros(n, dtype=bool)


## 11. 05 후보표와 CP 비교 기준

05에서 저장한 후보표를 사용함. CP의 동일 LOT 중앙값과 MAD 기반 변동 폭은 기존 v1의 사후 검토 기준임.
정상 수준 복귀를 양성으로 붙이지 않기 위한 조건이며, 실시간 정상 기준을 학습하는 단계는 아님.


In [ ]:
# 전체 LOT의 수준 차이를 양성으로 만들지 않고 국소 지속 후보만 사용함.
candidates = candidates.copy().reset_index(drop=True)

for column in ["START", "END"]:
    candidates[column] = pd.to_datetime(candidates[column], errors="raise")

time_array = times.to_numpy(dtype='datetime64[ns]')

cp_centers = {}
cp_sigmas = {}
cp_reference = raw.loc[~excluded, ['LOT_NO', 'CP-Z4']]

for lot, group in cp_reference.groupby('LOT_NO', sort=True):
    values = group['CP-Z4']

    center = values.median()

    cp_centers[lot] = center

    cp_sigmas[lot] = 1.4826 * (values - center).abs().median()

cp_centers = pd.Series(cp_centers)

cp_sigmas = pd.Series(cp_sigmas)


## 12. 라벨 근거로 사용할 정비 고르기

BM, CBM의 ELE, MAL, WER 또는 손상 부품교체가 있는 TBM을 후보로 삼음. 원인미상, 확인중, 이상없음 등의 기록은 양성 근거에서 뺌.
부품이 교체되지 않은 정비도 지우지 않음. 작업 중인 행은 따로 표시함.


In [ ]:
# WO별로 묶은 정비 표에서 라벨 근거로 사용할 정비를 고름.

maint = maintenance.loc[maintenance.EQP_TAG.eq(EQP_TAG)].copy()

for column in ["DETC_DT", "STRT_DT", "END_DT"]:
    maint[column] = pd.to_datetime(maint[column], errors="raise")

for column in ["MNT_TYPE", "FAIL_MODE", "REMARK", "PART_CHANGE_REASONS"]:
    maint[column] = maint[column].fillna("").astype(str).str.strip()

ambiguous = maint.REMARK.str.contains(AMBIGUOUS_REMARK, regex=True)

repair_fault = maint.MNT_TYPE.isin(["BM", "CBM"]) & maint.FAIL_MODE.isin(["ELE", "MAL", "WER"])

damaged_part = maint.MNT_TYPE.eq("TBM") & maint.HAS_PART_RECORD.eq(True) & maint.PART_CHANGE_REASONS.str.contains("손상", regex=False)

maint["WEAK_EVIDENCE_ELIGIBLE"] = (repair_fault | damaged_part) & ~ambiguous

eligible = maint.loc[maint.WEAK_EVIDENCE_ELIGIBLE]

for row in maint.itertuples(index=False):
    if pd.notna(row.STRT_DT) and pd.notna(row.END_DT):

        flags["MAINT_WORK_OVERLAP"] |= (times.ge(row.STRT_DT) & times.lt(row.END_DT)).to_numpy()


### 12-1. 정기수리와 강종변경 맥락 표시

정기수리 종료 날짜는 그 날까지 포함함. 일정과 전후 24시간은 계획 맥락일 뿐 실제 정지 구간이 아님.


In [ ]:
# 수리 날짜는 조업 맥락일 뿐 실제 정지시간이 아님. 날짜형 종료일은 그 날까지 포함함.
calendar = sources["repair"].loc[sources["repair"].PLANT_CD.eq("CR1")]

for row in calendar.itertuples(index=False):
    start = pd.Timestamp(row.STRT_DT).normalize()

    end = pd.Timestamp(row.END_DT).normalize() + pd.Timedelta(days=1)

    flags["SCHEDULED_REPAIR_CONTEXT"] |= (times.ge(start) & times.lt(end)).to_numpy()

    flags["REPAIR_PREPOST_24H_CONTEXT"] |= (
        (times.ge(start - pd.Timedelta(hours=24)) & times.lt(start)) |
        (times.ge(end) & times.lt(end + pd.Timedelta(hours=24)))
    ).to_numpy()

events = sources["event"].loc[sources["event"].PLANT_CD.eq("CR1")]

for stamp in pd.to_datetime(events.loc[events.EVT_TYPE.eq("강종변경"), "EVT_DT"]):
    flags["GRADE_CHANGE_NEARBY_30MIN"] |= times.between(stamp - pd.Timedelta(minutes=30), stamp + pd.Timedelta(minutes=30)).to_numpy()


## 13. 후보 구간을 원본 행에 연결

같은 LOT이고 시작~끝 시각 안에 있는 행 번호를 찾음. 각 표시가 붙는 범위를 직접 확인할 수 있게 함.


In [ ]:
candidate_positions = {}

for row in candidates.itertuples(index=False):
    hit = (raw['LOT_NO'].to_numpy() == row.LOT_NO)

    hit = hit & times.between(row.START, row.END).to_numpy()

    idx = np.flatnonzero(hit)

    idx = idx[np.argsort(time_array[idx])]

    candidate_positions[row.CANDIDATE_ID] = idx

    if row.SENSOR != "CP-Z4M":
        flags["ANY_SENSOR_CANDIDATE"][idx] = True

    if row.REVIEW_CATEGORY == "제어반응·운전조건 확인":
        flags["CONTROL_RESPONSE_CONTEXT"][idx] = True

    if row.PERIODIC_CONTEXT:
        flags["PERIODIC_CONTEXT"][idx] = True

    if row.LOT_BOUNDARY_CONTEXT:
        flags["LOT_BOUNDARY_CONTEXT"][idx] = True


### 13-1. 같은 존 TC 후보 확인

OP의 단독 변화와 TC 동반을 구분할 표시를 준비함. 다만 현재 v1은 제어반응 분류의 OP 후보를 양성으로 채택하지 않음.
기존 v1의 보류 조건과 라벨을 유지함.


In [ ]:
# OP 단독 변화는 양성으로 만들지 않고 같은 존 TC 지속 후보의 동시 관측을 요구함.
companion_masks = {}

for sensor in sensor_columns:
    if not sensor.startswith('OP-'):
        continue

    companion = sensor.replace("OP-", "TC-", 1)

    mask = np.zeros(n, dtype=bool)

    for row in candidates.loc[candidates.SENSOR.eq(companion)].itertuples(index=False):
        if pd.notna(row.AMPLITUDE) and pd.notna(row.THRESHOLD) and row.THRESHOLD > 0 and row.AMPLITUDE >= row.THRESHOLD:
            mask[candidate_positions[row.CANDIDATE_ID]] = True

    companion_masks[sensor] = mask


## 14. 후보의 강도와 정비 시간 연관 확인

후보 최대 이탈이 문턱의 2배 이상인지 확인함. 그 후보 안에서 감지 전 24시간 또는 작업 종료 후 24시간에 들어가는 실제 행만 찾음.
후보별 중간 결과를 candidate_checks에 보관해서 다음 셀에서 이어서 검토함.


In [ ]:
candidate_checks = []

for row in candidates.itertuples(index=False):
    idx = candidate_positions[row.CANDIDATE_ID]
    reasons = []

    if row.SENSOR not in sensor_columns:
        reasons.append("auxiliary_sensor_not_used")

    if row.KIND not in ["local_deviation_low_10s", "local_deviation_high_10s"]:
        reasons.append("not_local_sustained_candidate")

    strong = pd.notna(row.AMPLITUDE) and pd.notna(row.THRESHOLD) and row.THRESHOLD > 0 and row.AMPLITUDE >= AMPLITUDE_MULTIPLIER * row.THRESHOLD

    if not strong:
        reasons.append("below_2x_candidate_amplitude")

    if row.PERIODIC_CONTEXT:
        reasons.append("periodic_context_not_positive")

    if row.LOT_BOUNDARY_CONTEXT or row.REVIEW_CATEGORY == "LOT 경계 운전조건 확인":
        reasons.append("lot_boundary_context_not_positive")

    if row.REVIEW_CATEGORY == "제어반응·운전조건 확인":
        reasons.append("TC_OP_control_response_not_positive")

    if row.REVIEW_CATEGORY not in ["이상 검토 우선", "LOT 경계 운전조건 확인", "제어반응·운전조건 확인", "보조계측·생성관계 확인"]:
        reasons.append("unknown_review_category")

    domain_records = []

    domain_mask = np.zeros(len(idx), dtype=bool)

    for event in eligible.itertuples(index=False):
        local_times = times.iloc[idx]

        pre = np.zeros(len(idx), dtype=bool)

        if pd.notna(event.DETC_DT):
            pre = local_times.between(event.DETC_DT - pd.Timedelta(hours=ASSOCIATION_HOURS), event.DETC_DT).to_numpy()

        post = np.zeros(len(idx), dtype=bool)

        if pd.notna(event.END_DT):
            post = local_times.between(event.END_DT, event.END_DT + pd.Timedelta(hours=ASSOCIATION_HOURS)).to_numpy()

        if pre.any() or post.any():
            domain_mask |= pre | post

            domain_records.append((event, pre, post))

    if not domain_mask.any():
        reasons.append("no_eligible_maintenance_time_association")

    candidate_checks.append({'row': row, 'idx': idx, 'reasons': reasons,
                             'domain_records': domain_records, 'domain_mask': domain_mask})


## 15. 제외, 작업, 수리, CP 복귀 조건 적용

CP는 현재 값이 동일 LOT 기준선에서 충분히 멀 때만 남김. 기준선 계산이 안 되면 양성으로 사용하지 않음.
기간 전체를 1로 늘리는 대신 위에서 찾은 후보 행에만 조건을 적용함.


In [ ]:
for check in candidate_checks:
    row = check['row']
    idx = check['idx']
    reasons = check['reasons']

    usable = ~excluded[idx] & ~flags["MAINT_WORK_OVERLAP"][idx] & ~flags["SCHEDULED_REPAIR_CONTEXT"][idx] & ~flags["REPAIR_PREPOST_24H_CONTEXT"][idx]

    cp_baseline_rejected = 0

    if row.SENSOR == "CP-Z4":
        # 과거5분 기준선의 지연으로 생기는 정상 수준 복귀 후보를 제외함.
        # CP에만 적용하며 동일 LOT 전체 관측으로 만든 사후 약지도 가정임.
        center = cp_centers.get(row.LOT_NO, np.nan)

        sigma = cp_sigmas.get(row.LOT_NO, np.nan)

        if not np.isfinite(center) or not np.isfinite(sigma) or sigma <= 0:
            far = np.zeros(len(idx), dtype=bool)

            reasons.append("CP_reference_scale_unavailable")
        else:
            residual = pd.to_numeric(raw["CP-Z4"].iloc[idx], errors="coerce").to_numpy(dtype=float) - center

            limit = CP_LOT_SIGMA_MULTIPLIER * sigma

            # 높은 CP가 내려오는 중에도 기준선 밖이면 이탈은 남아 있음.
            # 국소 기울기 부호가 아니라 현재 수준의 절대편차로 복귀를 구분함.
            far = np.abs(residual) > limit

        cp_baseline_rejected = int((~far & ~excluded[idx]).sum())

        flags["CP_RETURN_TO_BASELINE_CONTEXT"][idx[~far & ~excluded[idx]]] = True

        usable &= far

        if not far.any():
            reasons.append("CP_return_or_small_change_at_LOT_baseline")

    if row.SENSOR in raw:
        usable &= np.isfinite(pd.to_numeric(raw[row.SENSOR].iloc[idx], errors="coerce").to_numpy(dtype=float))

    if row.SENSOR in companion_masks:
        usable &= companion_masks[row.SENSOR][idx]

        companion = row.SENSOR.replace("OP-", "TC-", 1)

        usable &= np.isfinite(pd.to_numeric(raw[companion].iloc[idx], errors="coerce").to_numpy(dtype=float))

        if not companion_masks[row.SENSOR][idx].any():
            reasons.append("OP_without_same_zone_TC_candidate")

    check['usable'] = usable
    check['cp_baseline_rejected'] = cp_baseline_rejected


## 16. 실제 1초 간격으로 10초 이상 이어지는 행만 남기기

관측 간격이 정확히 1초가 아니면 구간을 나눔. 10초 지속은 1초 간격 관측 11개 이상임.
중간에 빠진 시각을 메우거나 서로 다른 구간을 합치지 않음.


In [ ]:
for check in candidate_checks:
    idx = check['idx']
    reasons = check['reasons']
    runs = []

    if not reasons:
        selected = idx[check['usable'] & check['domain_mask']]

        if len(selected):
            gaps = np.diff(time_array[selected]).astype('timedelta64[ns]').astype('int64')

            cut_points = np.flatnonzero(gaps != 1_000_000_000) + 1

            for segment in np.split(selected, cut_points):
                if len(segment) >= MIN_ELAPSED_SECONDS + 1:
                    runs.append(segment)

    if runs:
        positive = np.concatenate(runs)
    else:
        positive = np.array([], dtype=int)

    if not len(positive) and not reasons:
        reasons.append('context_or_gap_leaves_less_than_10s')

    check['runs'] = runs
    check['positive'] = positive


## 17. 채택한 행에 정비번호, 센서, 후보 근거 모으기

같은 행에 여러 정비나 센서가 연결되면 집합으로 모아 중복을 없앰. 정비와 가깝다는 사실을 고장 원인으로 바꾸지는 않음.


In [ ]:
contributions = []
positive_evidence = {}

for check in candidate_checks:
    row = check['row']
    idx = check['idx']
    positive = check['positive']
    domain_records = check['domain_records']

    used_wo, used_type, assoc_kinds = set(), set(), set()

    for event, pre, post in domain_records:
        event_positions = idx[pre | post]

        hit = np.intersect1d(positive, event_positions)

        if not len(hit):
            continue

        used_wo.add(event.WO_NO)

        used_type.add(event.MNT_TYPE)

        for kind, local_mask in [("pre_detection_24h", pre), ("post_work_end_24h", post)]:
            if np.intersect1d(hit, idx[local_mask]).size:
                assoc_kinds.add(kind)

        for pos in hit:
            record = positive_evidence.setdefault(int(pos), {"wo": set(), "types": set(), "sensors": set(), "candidates": set(), "associations": set()})

            record["wo"].add(event.WO_NO)

            record["types"].add(event.MNT_TYPE)

            record["sensors"].add(row.SENSOR)

            record["candidates"].add(row.CANDIDATE_ID)

            if pd.notna(event.DETC_DT) and event.DETC_DT - pd.Timedelta(hours=ASSOCIATION_HOURS) <= times.iloc[pos] <= event.DETC_DT:
                record["associations"].add("pre_detection_24h")

            if event.END_DT <= times.iloc[pos] <= event.END_DT + pd.Timedelta(hours=ASSOCIATION_HOURS):
                record["associations"].add("post_work_end_24h")

        contributions.append({"LOT_NO": row.LOT_NO, "SENSOR": row.SENSOR, "MNT_TYPE": event.MNT_TYPE, "WO_NO": event.WO_NO, "POSITIONS": hit.tolist()})

    check['used_wo'] = used_wo
    check['used_type'] = used_type
    check['assoc_kinds'] = assoc_kinds


### 17-1. 후보별 채택, 미채택 결과 남기기

이 표는 라벨을 설명하는 검토표임. 후보의 범위와 실제 1인 구간을 구분해서 저장함.


In [ ]:
decisions = []

for check in candidate_checks:
    row = check['row']
    idx = check['idx']
    reasons = check['reasons']
    positive = check['positive']
    runs = check['runs']
    domain_records = check['domain_records']
    cp_baseline_rejected = check['cp_baseline_rejected']
    used_wo = check['used_wo']
    used_type = check['used_type']
    assoc_kinds = check['assoc_kinds']
    eligible_context_wo = []

    for event, pre, post in domain_records:
        eligible_context_wo.append(event.WO_NO)

    positive_run_texts = []

    for run in runs:
        start_text = times.iloc[run[0]].isoformat()

        end_text = times.iloc[run[-1]].isoformat()

        positive_run_texts.append(start_text + '~' + end_text)

    record = row._asdict()

    record.update({
        "WEAK_DECISION": "temporary_positive" if len(positive) else "not_positive_under_this_rule",
        "DECISION_CODES": "signal_and_eligible_record_time_association" if len(positive) else " | ".join(reasons),
        "SIGNAL_AMPLITUDE_RATIO": float(row.AMPLITUDE / row.THRESHOLD) if pd.notna(row.THRESHOLD) and row.THRESHOLD > 0 else None,
        "MATCHED_POSITIVE_ROWS": len(positive), "EXCLUDED_ROWS": int(excluded[idx].sum()),
        "WORK_OVERLAP_ROWS": int(flags["MAINT_WORK_OVERLAP"][idx].sum()),
        "REPAIR_CONTEXT_ROWS": int((flags["SCHEDULED_REPAIR_CONTEXT"][idx] | flags["REPAIR_PREPOST_24H_CONTEXT"][idx]).sum()),
        "CP_BASELINE_REJECTED_ROWS": cp_baseline_rejected,
        "CP_LOT_MEDIAN": float(cp_centers.get(row.LOT_NO, np.nan)) if row.SENSOR == "CP-Z4" else None,
        "CP_LOT_MAD_SIGMA": float(cp_sigmas.get(row.LOT_NO, np.nan)) if row.SENSOR == "CP-Z4" else None,
        "EVIDENCE_WO": " | ".join(sorted(used_wo)), "EVIDENCE_MNT_TYPE": " | ".join(sorted(used_type)),
        "ASSOCIATION_KIND": " | ".join(sorted(assoc_kinds)),
        "ELIGIBLE_CONTEXT_WO": " | ".join(eligible_context_wo),
        "POSITIVE_RUNS": " | ".join(positive_run_texts),
        "LABEL_STATUS": "임시 약지도; 센서·존 고장 원인 확정 아님", "MODEL_INPUT_ALLOWED": False,
    })

    decisions.append(record)


## 18. 원본 행 단위 임시 LABEL 붙이기

앞에서 채택한 행만 1로 바꿈. 나머지 비제외 행의 0은 규칙 미검출이며 정상 확정이 아님.
검토 표시와 근거는 LABEL을 설명하는 정보이므로 모델 입력에서 제외함.


In [ ]:
for pos, evidence in positive_evidence.items():
    output.at[pos, "LABEL"] = 1
    output.at[pos, "RULE"] = "strong_local_signal_AND_maintenance_time_association"
    output.at[pos, "LABEL_REASON"] = "강한 국소 신호 10초 이상과 유효 정비의 24시간 시간연관 동시 충족; 해당 센서 고장 원인 확정 아님"

    for column, key in [("EVIDENCE_WO", "wo"), ("EVIDENCE_MNT_TYPE", "types"), ("EVIDENCE_SENSOR", "sensors"), ("EVIDENCE_CANDIDATE_IDS", "candidates"), ("ASSOCIATION_KIND", "associations")]:
        output.at[pos, column] = " | ".join(sorted(evidence[key]))

for key, values in flags.items():
    output[key] = values

output["EVIDENCE_CONFIDENCE"] = np.where(output.LABEL.eq(1).fillna(False), "임시 시간연관; 원인 미확정", "규칙 미검출 또는 사용자 제외; 정상 확정 아님")

row_labels = output

interval_decisions = pd.DataFrame(decisions)


### 18-1. LOT, 정비유형, 센서별 기여도 계산

각 묶음 안에서는 고유 행 수를 셈. 여러 근거가 겹치면 묶음별 합계와 전체 양성 행 수는 다를 수 있음.


In [ ]:
contribution_tables = {}

for keys in [['LOT_NO'], ['MNT_TYPE'], ['SENSOR'], ['LOT_NO', 'MNT_TYPE', 'SENSOR']]:
    grouped_positions = {}

    for record in contributions:
        values = []

        for key in keys:
            values.append(record[key])

        group_key = tuple(values)

        if group_key not in grouped_positions:
            grouped_positions[group_key] = set()

        grouped_positions[group_key].update(record['POSITIONS'])

    counts = []

    for group_key, positions_in_group in sorted(grouped_positions.items()):
        count_row = {}

        for key, value in zip(keys, group_key):
            count_row[key] = value

        count_row['POSITIVE_ROWS'] = len(positions_in_group)

        counts.append(count_row)

    contribution_tables[tuple(keys)] = counts

sensor_candidate_counts = {}

for sensor in sensor_columns:
    sensor_candidate_counts[sensor] = int((candidates['SENSOR'] == sensor).sum())

cp_reference_rows = []

for lot in cp_centers.index:
    cp_reference_rows.append({'LOT_NO': int(lot), 'MEDIAN': float(cp_centers.loc[lot]),
                              'MAD_SIGMA': float(cp_sigmas.loc[lot])})

event_type_counts = {}

for event_type, count in events['EVT_TYPE'].value_counts().items():
    event_type_counts[str(event_type)] = int(count)

forbidden_columns = []

for column in output.columns:
    if column not in ['LOT_NO', 'MEAS_DT', 'GRADE', 'LABEL']:
        forbidden_columns.append(column)


### 18-2. v1의 판정 조건과 한계 기록

기존 v1의 라벨 정책을 그대로 기록함. 통계 문턱과 사후 정비 근거로 만든 약지도이므로 실제 고장 정답이나 실시간 탐지 성능을 뜻하지 않음.


In [ ]:
label_policy = {
    "version": VERSION, "label_status": "사용자 선택한 첫 실험용 임시 약지도; 현장 정답 아님",
    "source_rows": n, "excluded_rows": int(excluded.sum()), "retained_rows": int((~excluded).sum()),
    "positive_rows": int(output.LABEL.eq(1).sum()), "negative_rows": int(output.LABEL.eq(0).sum()),
    "zero_definition": "21계측 신호+유효 정비 시간연관 규칙 미검출; 불확실 후보·운전 맥락도 포함하므로 정상 확정이 아님",
    "one_definition": "강한 국소 지속 신호와 적격 정비의 감지 전/작업 종료 후 시간연관 AND; 센서 고장 원인 확정이 아님",
    "candidate_count": len(candidates), "sensor_columns_reviewed": sensor_columns,
    "sensor_candidate_counts": sensor_candidate_counts,
    "periodic_context_true_candidates": int(candidates.PERIODIC_CONTEXT.sum()),
    "amplitude_multiplier": AMPLITUDE_MULTIPLIER, "association_hours": ASSOCIATION_HOURS,
    "cp_lot_sigma_multiplier": CP_LOT_SIGMA_MULTIPLIER,
    "cp_baseline_rule": "CP만 사용자 제외 후 동일 LOT 중앙값과 현재값의 절대편차가 12×1.4826×MAD를 넘는 실제 행을 요구하고 다시 1초간격10초 지속 확인. 5분 기준선 지연에 따른 기준선 복귀 후보를 1로 만들지 않음. 국소 감소 중이어도 현재 수준이 기준선에서 멀면 이탈로 남음",
    "cp_reference_by_lot": cp_reference_rows,
    "min_elapsed_seconds": MIN_ELAPSED_SECONDS, "min_observations": MIN_ELAPSED_SECONDS + 1,
    "threshold_status": "2배 문턱·CP12MADσ·24시간·수리전후24시간은 탐색 가정이며 공정허용치가 아님. 행수 목표 없음",
    "maintenance_eligible_count": len(eligible), "maintenance_eligible_wo": eligible.WO_NO.tolist(),
    "maintenance_rule": "BM/CBM의 ELE·MAL·WER 또는 TBM 손상교체; 원인미상·확인중·이상없음·재현안됨·재점검필요 제외",
    "positive_contribution_by_lot": contribution_tables[('LOT_NO',)],
    "positive_contribution_by_type": contribution_tables[('MNT_TYPE',)],
    "positive_contribution_by_sensor": contribution_tables[('SENSOR',)],
    "positive_contribution_by_lot_type_sensor": contribution_tables[('LOT_NO', 'MNT_TYPE', 'SENSOR')],
    "contribution_note": "각 그룹 내부는 고유 행 수. 여러 센서/WO 근거가 겹치면 그룹 합계는 전체 양성 수와 다를 수 있음",
    "event_types": event_type_counts,
    "limitations": [
        "정비는 CAL01 설비 단위이며 특정 존·센서 귀속이 없음. 근접 기록은 인과 근거가 아님",
        "모든 21계측을 검토하지만 후보가 없는 센서가 정상으로 검증됐다는 뜻은 아님. CP-Z4M은 의미 미확인 보조계측으로 제외",
        "CP도 단독 신호로 1이 되지 않고 적격 정비 근거가 필요함. OP는 같은 존 TC 지속 후보와 동시 10초 관측을 추가 요구",
        "TC-Z3/OP-Z3 제어반응 분류·LOT경계·주기 후보·작업 중·수리일정/전후24시간은 양성으로 채택하지 않음. 사용자 제외가 아니므로 이 행들은 규칙 미검출 0",
        "수리 종료 날짜는 날짜 포함으로 처리했지만 실제 정지구간이 아님. DOWN_MIN으로 정지시점을 역산하지 않음",
        "신호 2배 조건은 후보 최대이탈의 강도임. 채택한 각 행이 2배 문턱을 넘는다는 뜻은 아님",
        "후보 생성의 LOT 통계·사후 정비 시각은 미래정보일 수 있음. 모든 근거·판정·문맥 플래그는 X에서 제외",
        "CP 기준선은 강종 내 동일 LOT의 전체 유효 관측을 사용한 사후 약지도이며 운전단계 혼합에 오염될 수 있음. 현장 정상 분포나 실시간 기준선으로 보증하지 않음. TC 등 다른 센서에 12MAD 조건을 일괄 적용하지 않음",
        "전체 LOT 수준 차이는 양성 규칙으로 사용하지 않음. 라벨 0에 남은 미해결 사례도 존재함",
    ],
    "model_input_forbidden": forbidden_columns,
}


## 19. 원본 관측 행에 과거 특징 연결

앞에서 계산한 과거 특징 38개를 같은 LOT, 분 시각으로 붙임. 원본 1초 행은 유지하고 사용자 제외 행의 특징은 비워둠.


In [ ]:
DATASET_VERSION = 'v1'
KEYS = ['LOT_NO', 'MEAS_DT']
TARGET_SENSORS = ['TC-Z3', 'OP-Z3', 'TC-Z4', 'OP-Z4', 'TC-Z5', 'OP-Z5', 'CP-Z4']
PAST_SUFFIXES = ['1분전', '5분전', '직전10분평균', '직전10분표준편차', '직전두분차이']
PAST_FEATURES = []

for sensor in TARGET_SENSORS:
    for suffix in PAST_SUFFIXES:
        PAST_FEATURES.append(sensor + '_' + suffix)

for sensor in ['OP-Z3', 'OP-Z4', 'OP-Z5']:
    PAST_FEATURES.append(sensor + '_직전10분포화율')

raw = temp
lots = lot_context

master = marked.drop(columns='GRADE').merge(lots[['LOT_NO', 'GRADE', 'START']], on='LOT_NO', validate='many_to_one')

master['LOT_ELAPSED_MIN'] = (master['MEAS_DT'] - master['START']).dt.total_seconds() / 60

master = master.drop(columns='START')

master['FEATURE_MINUTE'] = master['MEAS_DT'].dt.floor('min')

past = features[KEYS + ['강종'] + PAST_FEATURES].rename(columns={'MEAS_DT': 'FEATURE_MINUTE', '강종': 'FEATURE_GRADE'})

master = master.merge(past, on=['LOT_NO', 'FEATURE_MINUTE'], how='left', validate='many_to_one')

master.loc[master['EXCLUDED'], PAST_FEATURES] = np.nan

master = master.drop(columns='FEATURE_GRADE')


## 20. 라벨, 근거를 붙이고 원본 행 수 유지

라벨은 LOT, 시각 키로 1:1 연결함. 근거 정비의 부품, 정지 분량은 문자열로 남겨 여러 부품 때문에 원본 행이 늘지 않게 함.


In [ ]:
metadata_columns = []

for column in row_labels.columns:
    if column not in master.columns and column not in KEYS:
        metadata_columns.append(column)

master = master.merge(row_labels[KEYS + metadata_columns], on=KEYS, how="left", validate="one_to_one", indicator=True)

master = master.drop(columns="_merge")

master = master.rename(columns={"MODEL_INPUT_ALLOWED": "EVIDENCE_INPUT_ALLOWED"})

# 라벨 근거 WO에 연결된 정비, 교체 값을 남김. 여러 부품 때문에 센서 행이 늘지 않게 함.
event_index = maintenance.set_index("WO_NO")

for field in ["DOWN_MIN", "HAS_PART_RECORD", "PART_NAMES", "PART_QTY_TOTAL", "PART_CHANGE_REASONS", "FAIL_MODE"]:
    summaries = {}

    for value in master.EVIDENCE_WO.fillna("").unique():
        events = value.split(" | ") if value else []

        values_for_wo = []

        for wo in events:
            values_for_wo.append(str(wo) + ':' + str(event_index.at[wo, field]))

        summaries[value] = ' | '.join(values_for_wo)

    master["EVIDENCE_" + field] = master.EVIDENCE_WO.fillna("").map(summaries)

master.loc[master.EXCLUDED, "LABEL"] = pd.NA

master["LABEL"] = master.LABEL.astype("Int8")

links = row_links[KEYS + ["LOT_CONTEXT_ID", "SENSOR_CANDIDATE_IDS"]]

master = master.merge(links, on=KEYS, how="left", validate="one_to_one")

for column in ["LOT_CONTEXT_ID", "SENSOR_CANDIDATE_IDS"]:
    master[column] = master[column].fillna("")


## 21. RF 수치 입력 후보와 식별, 근거 분리

원본 21계측 + 과거 특징 38개 + 경과시간 1개 + 강종 원핫 7개임. RF 입력 후보일 뿐 학습한 모델은 아님.
FEATURES_COMPLETE는 67개가 모두 유한한 숫자인지, MODEL_ELIGIBLE은 제외되지 않고 입력, LABEL이 있는지 표시함.


In [ ]:
sensors = []

for column in raw.columns:
    if column not in KEYS and column != 'CP-Z4M':
        sensors.append(column)

input_columns = sensors + ["LOT_ELAPSED_MIN"] + PAST_FEATURES

for grade in sorted(lots.GRADE.unique()):
    column = "GRADE_" + grade

    master[column] = master.GRADE.eq(grade).astype("int8")

    input_columns.append(column)

master["FEATURES_COMPLETE"] = np.isfinite(master[input_columns].to_numpy(dtype=float)).all(axis=1)

master["MODEL_ELIGIBLE"] = ~master.EXCLUDED & master.FEATURES_COMPLETE & master.LABEL.notna()

master = master.sort_values(KEYS).reset_index(drop=True)

master.insert(0, "ROW_ID", master.LOT_NO.astype(str) + "_" + master.MEAS_DT.dt.strftime("%Y%m%dT%H%M%S"))

ordered_columns = []

for column in master.columns:
    if column != 'LABEL':
        ordered_columns.append(column)

master = master[ordered_columns + ['LABEL']]

print('기준표:', master.shape)
print('LABEL:', master['LABEL'].value_counts(dropna=False).to_dict())


## 22. 입력 파일과 생성 노트북 기록

사용한 파일의 해시와 이 노트북 경로를 기록함. 기존 v1의 입력 기록을 유지하며 개인 모듈의 계산 함수는 불러오지 않음.


In [ ]:
source_files = [TEMP_CSV, EVENT_CSV, REPAIR_CSV, EQP_CSV, PART_CSV, MAINT_CSV, PARTCHG_CSV,
                FEATURE_FILE, CANDIDATE_FILE, ROW_LINK_FILE,
                os.path.join(ROOT, 'src', 'namelesspark_weak_labels.py'),
                os.path.join(ROOT, 'src', 'namelesspark_datasets.py'),
                os.path.join(ROOT, 'src', 'namelesspark_pipeline.py'),
                os.path.join(ROOT, 'src', 'namelesspark_temporal_review.py')]

source_sha256 = {}

for file_path in source_files:
    file_hash = hashlib.sha256()

    with open(file_path, 'rb') as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            file_hash.update(chunk)

    relative_path = os.path.relpath(file_path, ROOT).replace('\\', '/')

    source_sha256[relative_path] = file_hash.hexdigest()

generator_notebook = 'notebooks/namelesspark/06_labeled_dataframe.ipynb'

generator_path = os.path.join(ROOT, 'notebooks', 'namelesspark', '06_labeled_dataframe.ipynb')

generator_hash = hashlib.sha256()

with open(generator_path, 'rb') as file:
    while True:
        chunk = file.read(1024 * 1024)

        if not chunk:
            break

        generator_hash.update(chunk)

MASTER_NAME = 'cal01_base_' + DATASET_VERSION + '.csv'

master_path = os.path.join(PROCESSED, MASTER_NAME)
meta_dir = os.path.join(PROCESSED, 'metadata', DATASET_VERSION)
manifest_path = os.path.join(meta_dir, 'dataset.json')

eligible_label_counts = {}

for label, count in master.loc[master['MODEL_ELIGIBLE'], 'LABEL'].value_counts().items():
    eligible_label_counts[str(int(label))] = int(count)

raw_measurement_columns = []

for column in master.columns:
    if column.startswith(('TC-', 'OP-')) and '_' not in column:
        raw_measurement_columns.append(column)

raw_measurement_columns += ['CLN-IN', 'CP-Z4', 'CP-Z4M']


### 22-1. 기준표의 입력과 라벨 정책 기록

입력 파일 해시, 라벨 정책, 입력 후보 목록을 manifest에 기록함.
검토 근거와 미래 정비 정보는 X가 아니며, 분할, 9:1 추출, 학습은 아직 하지 않음.


In [ ]:
manifest = {
    'dataset_version': DATASET_VERSION, 'master_file': MASTER_NAME,
    'row_key': ['ROW_ID', 'LOT_NO', 'MEAS_DT'], 'rows': len(master), 'columns': len(master.columns),
    'label_0': int(master['LABEL'].eq(0).sum()), 'label_1': int(master['LABEL'].eq(1).sum()),
    'excluded_rows': int(master['EXCLUDED'].sum()), 'label_na': int(master['LABEL'].isna().sum()),
    'model_eligible_rows': int(master['MODEL_ELIGIBLE'].sum()),
    'eligible_label_counts': eligible_label_counts, 'input_columns': input_columns,
    'raw_measurement_columns': raw_measurement_columns, 'label_policy': label_policy,
    'input_semantics': '현재 관측과 현재 분 이전에 완성된 과거 특징만 사용. LABEL·근거·사후 사건은 X에서 제외함.',
    'split_status': 'not_assigned', 'sampling_applied': False, 'models_trained': False,
    'sampling_policy': '시간·LOT로 먼저 분할하고 학습 부분에만 정상:이상=9:1 적용. 검증·시험은 원래 분포 유지.',
    'source_sha256': source_sha256,
    'generator_notebook': generator_notebook,
    'generator_notebook_sha256': generator_hash.hexdigest(),
}


## 23. 기준표와 검토 결과 저장

앞에서 계산한 표를 저장함. 기준 CSV는 하나이며 분류, 회귀 폴더에는 같은 표를 읽는 정의를 둠.
원본 데이터 파일은 쓰지 않음.


In [ ]:
os.makedirs(meta_dir, exist_ok=True)
os.makedirs(REVIEW, exist_ok=True)

master.to_csv(master_path, index=False, encoding=ENC)

master_hash = hashlib.sha256()

with open(master_path, 'rb') as file:
    while True:
        chunk = file.read(1024 * 1024)

        if not chunk:
            break

        master_hash.update(chunk)

manifest['master_sha256'] = master_hash.hexdigest()

with open(manifest_path, 'w', encoding='utf-8') as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

interval_path = os.path.join(REVIEW, 'weak_label_interval_decisions.csv')

interval_decisions.to_csv(interval_path, index=False, encoding=ENC)


### 23-1. 분류, 회귀 입력 정의 저장

67개는 RF의 수치 입력 후보임. Z-score는 08에서 별도로 검토하며 IF 입력도 아직 미정임.
회귀는 목표 센서와 미래 시차를 정하지 않았으므로 y를 만들지 않음.


In [ ]:
for task in ['classification', 'regression']:
    directory = os.path.join(PROCESSED, task, DATASET_VERSION)

    os.makedirs(directory, exist_ok=True)

    contract = {
        'dataset_version': DATASET_VERSION,
        'master_relative_path': '../../' + MASTER_NAME,
        'master_sha256': manifest['master_sha256'], 'input_columns': input_columns,
        'row_filter': 'MODEL_ELIGIBLE == True',
        'split_status': 'not_assigned', 'sampling_applied': False,
    }

    if task == 'classification':
        contract['target'] = 'LABEL'
        contract['label_kind'] = 'weak_domain_assisted_not_ground_truth'
        contract['methods'] = ['Z-score', 'IsolationForest', 'RandomForestClassifier']
        contract['method_input_columns'] = {'RandomForestClassifier': input_columns, 'IsolationForest': None, 'Z-score': None}
        contract['method_input_status'] = {
            'RandomForestClassifier': 'candidate_columns_only_not_trained_or_validated',
            'IsolationForest': 'input_selection_pending',
            'Z-score': 'input_selection_grouping_and_threshold_pending',
        }
        contract['evaluation'] = '같은 기준표와 평가 행을 공유할 수 있으나 방법별 입력 검토가 남아 있음. LABEL은 X와 Z-score·IF fit에서 제외함.'
    else:
        contract['target_sensor'] = None
        contract['horizon_seconds'] = None
        contract['status'] = 'target_not_selected'
        contract['target_rule'] = '같은 LOT의 정확한 미래 시각을 연결. 미래값은 X에서 제외하며 사용자 제외·분할 경계를 넘는 정답은 사용하지 않음.'

    contract_path = os.path.join(directory, 'dataset.json')

    with open(contract_path, 'w', encoding='utf-8') as file:
        json.dump(contract, file, ensure_ascii=False, indent=2)

print('공통 기준:', master_path)
print('전체:', len(master), '/ 임시 0:', manifest['label_0'], '/ 임시 1:', manifest['label_1'])
print('사용자 제외:', manifest['excluded_rows'], '/ RF 입력 후보가 모두 있는 행:', manifest['model_eligible_rows'])
print('기준 통계, 점수, 분할, 샘플링, 모델 학습은 실행하지 않았음')
